In [ ]:
deepseek_path='/content/drive/MyDrive/TFM_Jorge_MIAA_24-25/6.IJIMAI_Review/4. Pipeline/deepseek_finetuned_model_multi_mit'
dataset_path='/content/drive/MyDrive/TFM_Jorge_MIAA_24-25/6.IJIMAI_Review/2. Latency and rt inference/DNN-EdgeIIoT-dataset.csv'

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# Load tockenizer
tokenizer = AutoTokenizer.from_pretrained(
    deepseek_path,
    fix_mistral_regex=True,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    deepseek_path,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map=None,
)

model.to(DEVICE)
model.eval()

print("Model and tokenizer loaded correctly!")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

Model and tokenizer loaded correctly!


In [79]:
def build_joint_prompt_from_row(row):
    return (
        "Below is an instruction that describes a task, paired with an input that provides further context. "
        "Write a response that appropriately completes the request. Before answering, think carefully about the question "
        "and create a step-by-step chain of thoughts to ensure a logical and accurate response.\n"
        "### Instruction:\n"
        "You are a cybersecurity expert specializing in IoT security. Your task is to analyze network logs and determine "
        "whether the given log data indicates a potential attack. Only provide the type of attack if it is an attack. "
        "If the log data is normal traffic, state that it is normal traffic. If it is an attack, also include 4 to 5 specific technical mitigations.\n"
        "### Question:\n"
        f"- The length of the DNS query is: {row['dns.qry.name.len']}\n"
        f"- The MQTT protocol name used is: {row['mqtt.protoname']}\n"
        f"- The MQTT message type is: {row['mqtt.msg']}\n"
        f"- The MQTT topic is: {row['mqtt.topic']}\n"
        f"- The MQTT connection acknowledgment flags are: {row['mqtt.conack.flags']}\n"
        f"- TCP options set in the packet are: {row['tcp.options']}\n"
        f"- TCP destination port is: {row['tcp.dstport']}\n"
        "### Response:\n"
    )


In [ ]:
label_set = [
    "DDoS_UDP", "DDoS_ICMP", "SQL_injection", "Password",
    "Vulnerability_scanner", "DDoS_TCP", "DDoS_HTTP", "Uploading", "Backdoor",
    "Port_Scanning", "XSS", "Ransomware", "MITM", "Fingerprinting"
]

def extract_attack_label(output_text: str) -> str:
    text_lower = output_text.lower()
    # First, we try to detect normal traffic
    if "this log data is normal traffic." in text_lower:
        return "Normal"
    # Then, we check for each attack label
    for label in label_set:
        if label.lower() in text_lower:
            return label
    return "Unknown"


In [ ]:
import re

def extract_mitigations(output_text: str) -> str:
    m_think = re.search(r"<think>(.*)</think>", output_text, flags=re.DOTALL | re.IGNORECASE)
    text = m_think.group(1) if m_think else output_text

    #  Search for the "Mitigations:" section
    m_mit = re.search(r"Mitigations:\s*(.*)", text, flags=re.DOTALL | re.IGNORECASE)
    if not m_mit:
        return "" 

    mitigations_block = m_mit.group(1).strip()
    return mitigations_block


In [ ]:
def infer_attack_and_mitigations(prompt, max_new_tokens=1000):
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True, 
            top_p=0.9,
            temperature=0.7,
        )
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    attack = extract_attack_label(full_text)
    mitigations = extract_mitigations(full_text)

    return attack, mitigations, full_text 


In [ ]:
def run_pipeline_on_log(row):
    # 1.- Build prompt from log row
    prompt = build_joint_prompt_from_row(row)

    # 2.- Infer attack and mitigations
    attack_pred, mitigations_text, _ = infer_attack_and_mitigations(prompt)

    # 3.- Return structured result
    if attack_pred == "Normal":
        mitigations_text = None
    return {
        "attack_prediction": attack_pred,
        "mitigations": mitigations_text
    }


In [ ]:
import pandas as pd
import numpy as np
# Load dataset
df = pd.read_csv(dataset_path, low_memory=False)

# 21k sampleas random
df_sampled = df.sample(n=21000, random_state=42).reset_index(drop=True)

In [ ]:
import numpy as np

# Get unique labels
unique_labels = df_sampled["Attack_type"].unique()
np.random.seed(42)

for label in unique_labels:
    # Filter rows with the current label
    subset = df_sampled[df_sampled["Attack_type"] == label]

    if len(subset) == 0:
        continue 

    # Select a random sample
    row = subset.sample(n=1).iloc[0]

    # Run pipeline
    result = run_pipeline_on_log(row)

    print("\n" + "="*90)
    print(f"REAL LABEL:      {label}")
    print(f"PREDICTED LABEL: {result['attack_prediction']}")
    print("MITIGATIONS:\n", result["mitigations"])
    print("="*90 + "\n")


REAL LABEL:      DDoS_ICMP
PREDICTED LABEL: MITM
MITIGATIONS:
 - Enforce strict certificate validation for all IoT devices and gateways during communication establishment.
- Implement mutual TLS (mTLS) for all IoT device connections requiring authentication and encryption.
- Use Elliptic Curve Cryptography (ECC) for key exchange in IoT device configurations to enable stronger authentication and reduced key sizes.
- Update firmware and software of IoT devices immediately when security patches are available to fix potential vulnerabilities exploited by this attack.


REAL LABEL:      DDoS_UDP
PREDICTED LABEL: Fingerprinting
MITIGATIONS:
 - Remove all unnecessary diagnostic logging in IoT edge devices to prevent revealing implementation details via system outputs.
- Implement strict access controls on device APIs to only allow authorized queries and log access attempts.
- Use randomized identifiers instead of static ones for IoT devices to avoid consistent fingerprints being created.
- A